# 01 Main Results Table

LaTeX position:

- `tab:main-overall`

이 notebook은 main table과 간단한 summary design을 같이 본다.
현재 값은 paper draft에 맞춘 preview다.


In [ ]:
from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path('/workspace/FeaturedMoE/writing/260418_final_exp_figure')
DATA_DIR = ROOT / 'data'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import slot_viz_helpers as viz
importlib.reload(viz)

from slot_viz_helpers import (
    category_bar_line_plot,
    dual_metric_grouped_plot,
    heatmap_panel,
    line_panel,
    scatter_panel,
    setup_style,
    single_metric_bar,
    style_plain_table,
    style_ranked_table,
)

setup_style()


def pivot_metric_frame(df, id_cols, metric_map):
    wide = df.pivot_table(index=id_cols, columns=['metric', 'cutoff'], values='value', aggfunc='first').reset_index()
    flattened = []
    for col in wide.columns:
        if isinstance(col, tuple):
            left, right = col
            if right == '':
                flattened.append(left)
            elif left == '':
                flattened.append(str(right))
            else:
                flattened.append(f"{left}_{right}")
        else:
            flattened.append(col)
    wide.columns = flattened
    rename_map = {}
    for new_name, (metric, cutoff) in metric_map.items():
        rename_map[f"{metric}_{cutoff}"] = new_name
    wide = wide.rename(columns=rename_map)
    for new_name in metric_map:
        if new_name not in wide.columns:
            wide[new_name] = np.nan
    return wide


def show_status_notes(df, placeholder_note=None, ready_note=None):
    if 'status' not in df.columns:
        return
    status_series = df['status'].dropna().astype(str)
    if status_series.empty:
        return
    if status_series.str.contains('placeholder', case=False).any() and placeholder_note:
        display(Markdown(placeholder_note))
    elif ready_note:
        display(Markdown(ready_note))

table_df = pd.read_csv(DATA_DIR / '01_main_results_table.csv')
metric_plot_df = pd.read_csv(DATA_DIR / '01_main_results_plot.csv')

display(Markdown('### Main table preview'))
display(style_ranked_table(
    table_df,
    numeric_columns=['SASRec','GRU4Rec','TiSASRec','FEARec','DuoRec','BSARec','FAME','DIF-SR','FDSA','RouteRec'],
    lower_is_better=False,
    caption='Main experimental results preview',
))


In [ ]:
display(Markdown('### Summary metric design'))
fig, ax = plt.subplots(figsize=(11.5, 4.8), constrained_layout=True)
dual_metric_grouped_plot(
    metric_plot_df,
    category_col='dataset',
    variant_col='variant',
    bar_col='ndcg20',
    line_col='hr10',
    ax=ax,
    title='Main comparison summary',
    bar_label='NDCG@20',
    line_label='HR@10',
    category_order=['Beauty', 'Foursquare', 'KuaiRec', 'Retail Rocket'],
    variant_order=['SASRec', 'BSARec', 'DuoRec', 'RouteRec'],
    rotate=25,
)
plt.show()
